# Crear una muestra filtrando paises

`TAMANO_MUESTRA` define el total de filas/imagenes de salida. `MIN_FILAS_PAIS` es exclusivamente un umbral para filtrar paises: se admiten los codigos cuya `cantidad` en `assets/countries_count.csv` sea mayor o igual al umbral. No se exige ese minimo en la muestra final.

Con `BALANCEADO = True`, se asigna la misma cantidad por pais, con una diferencia maxima de una fila cuando el total no es divisible por la cantidad de paises. Si no hay suficientes filas locales para mantener ese balance sin reemplazo, se informa el maximo permitido y se detiene la ejecucion. Con `BALANCEADO = False`, se reparten los cupos proporcionalmente a las filas realmente disponibles, usando restos mayores para obtener el total exacto. Luego se seleccionan filas aleatorias sin reemplazo por pais, de forma reproducible con `SEMILLA`. Un pais admitido puede recibir cero filas si la muestra es pequena. Los paises ausentes del CSV y las filas sin pais se excluyen.

Se conservan todas las columnas, incluidas las imagenes, sin decodificarlas. El muestreo total selecciona filas/imagenes, por lo que no necesariamente incluye todas las vistas de cada panorama original. Despues se divide esa muestra en train y validacion: todas las vistas seleccionadas de un `panoid` quedan juntas. `TAMANO_MUESTRA`, `MIN_FILAS_PAIS` y `BALANCEADO` se aplican al conjunto completo, antes de separarlo.

`FRACCION_VALIDACION = 0.20` configura el objetivo de validacion por pais. Se barajan panoramas dentro de cada pais y se elige una cantidad de grupos cercana a ese objetivo. El porcentaje real es aproximado porque los grupos no se dividen. Si un pais tiene un solo panorama seleccionado, queda en train y se informa. Los panoid faltantes o asociados a varios paises en la muestra producen un error. La primera pasada lee `country_code` y `panoid`; la segunda copia las imagenes por lotes.

In [18]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from tqdm.auto import tqdm

# Parámetros editables.
TAMANO_MUESTRA = 20_000
MIN_FILAS_PAIS = 1000  # Umbral sobre cantidad en el CSV, no sobre la muestra.
CSV_CONTEOS = "src/notebooks/preprocessing/assets/countries_count.csv"
BALANCEADO = True  # True: cupos iguales (+/- 1); False: proporcionales.
FRACCION_VALIDACION = 0.20  # Entre 0 y 1, aplicada a la muestra completa.
SEMILLA = 42
PATRON_ARCHIVOS = "**/train-*.parquet"  # Solo train; usar "**/*.parquet" para todos los splits.
CARPETA_SALIDA = "datasets/samples"  # Relativa a la raíz del proyecto, o absoluta.
TAMANO_LOTE = 128  # Limita la memoria al leer imágenes.

## Localizar el dataset y el CSV

`DATA_URL` se toma del entorno si ya esta definida; de lo contrario se lee del `.env`. Las rutas relativas del dataset, del CSV y de la salida se resuelven respecto de la carpeta del `.env`. El CSV decide que paises participan; el patron de archivos decide de que Parquet se extraen sus filas.

In [19]:
def entero_valido(valor, nombre, minimo):
    if isinstance(valor, bool) or not isinstance(valor, (int, np.integer)) or valor < minimo:
        raise ValueError(f"{nombre} debe ser un entero >= {minimo}.")

for nombre in ("TAMANO_MUESTRA", "MIN_FILAS_PAIS", "TAMANO_LOTE"):
    entero_valido(globals()[nombre], nombre, 1)
entero_valido(SEMILLA, "SEMILLA", 0)
if (isinstance(FRACCION_VALIDACION, bool)
        or not isinstance(FRACCION_VALIDACION, (int, float))
        or not 0 < FRACCION_VALIDACION < 1):
    raise ValueError("FRACCION_VALIDACION debe ser un numero entre 0 y 1, sin incluir extremos.")
if not isinstance(BALANCEADO, bool):
    raise ValueError("BALANCEADO debe ser True o False.")

env_path = next(
    (carpeta / ".env" for carpeta in (Path.cwd(), *Path.cwd().parents)
     if (carpeta / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("No se encontró el .env del proyecto desde el directorio actual.")

valor_env = ""
for linea in env_path.read_text(encoding="utf-8-sig").splitlines():
    clave, separador, valor = linea.partition("=")
    if separador and clave.strip() == "DATA_URL":
        valor_env = valor.strip().strip("\"'")

ruta_configurada = os.environ.get("DATA_URL", valor_env).strip()
if not ruta_configurada:
    raise ValueError("Configurá DATA_URL en el .env con la carpeta del dataset local.")

raiz_proyecto = env_path.parent
ruta_datos = Path(ruta_configurada).expanduser()
if not ruta_datos.is_absolute():
    ruta_datos = raiz_proyecto / ruta_datos
ruta_datos = ruta_datos.resolve()
if not ruta_datos.is_dir():
    raise NotADirectoryError(f"DATA_URL apunta a una carpeta inexistente: {ruta_datos}")

archivos = sorted(p for p in ruta_datos.glob(PATRON_ARCHIVOS) if p.is_file())
if not archivos:
    raise FileNotFoundError(f"No se encontraron Parquet con {PATRON_ARCHIVOS!r} en {ruta_datos}")

raiz_salida = Path(CARPETA_SALIDA).expanduser()
if not raiz_salida.is_absolute():
    raiz_salida = raiz_proyecto / raiz_salida
raiz_salida = raiz_salida.resolve()
# Evitar incorporar muestras previas como fuente en ejecuciones posteriores.
if raiz_salida == ruta_datos or ruta_datos in raiz_salida.parents:
    raise ValueError("CARPETA_SALIDA debe estar fuera de la carpeta del dataset de entrada.")

print(f"Dataset: {ruta_datos}")
print(f"Archivos de entrada: {len(archivos):,}")

ruta_csv = Path(CSV_CONTEOS).expanduser()
if not ruta_csv.is_absolute():
    ruta_csv = raiz_proyecto / ruta_csv
ruta_csv = ruta_csv.resolve()
# Preservar NA (Namibia): pandas lo interpreta como faltante por defecto.
conteos_csv = pd.read_csv(ruta_csv, dtype={"pais": "string"}, keep_default_na=False)
if not {"pais", "cantidad"}.issubset(conteos_csv.columns):
    raise ValueError("El CSV debe contener las columnas pais y cantidad.")
conteos_csv = conteos_csv[["pais", "cantidad"]].copy()
conteos_csv["pais"] = conteos_csv["pais"].str.strip().str.upper().replace("", pd.NA)
conteos_csv["cantidad"] = pd.to_numeric(conteos_csv["cantidad"], errors="raise")
if (conteos_csv.isna().any().any() or (conteos_csv["cantidad"] < 0).any()
        or (conteos_csv["cantidad"] % 1 != 0).any()):
    raise ValueError("El CSV debe tener codigos presentes y cantidades enteras no negativas.")
# TR figura dos veces con el mismo conteo: no sumar ni duplicar ese pais.
conteos_csv = conteos_csv.drop_duplicates()
if conteos_csv["pais"].duplicated().any():
    raise ValueError("El CSV tiene conteos contradictorios para un mismo pais.")
conteos_csv = conteos_csv.set_index("pais")["cantidad"].astype("int64").sort_index()
paises_admitidos = conteos_csv.index[conteos_csv >= MIN_FILAS_PAIS]
if paises_admitidos.empty:
    raise ValueError("Ningun pais del CSV alcanza MIN_FILAS_PAIS.")
print(f"Paises admitidos por el CSV: {len(paises_admitidos)}")

Dataset: C:\Users\jurad\Documents\facultad\rrnn2\tp-final\world-streetview-500k\data
Archivos de entrada: 393
Paises admitidos por el CSV: 71


## Calcular cupos entre los paises admitidos

Se usa un indice global de filas sobre los archivos ordenados. Solo se mantiene en memoria la columna de paises. El resumen compara el conteo del CSV, las filas disponibles en los Parquet seleccionados y las filas de salida. Un pais admitido segun el CSV sigue siendo elegible aunque haya menos filas locales que el umbral.

In [20]:
partes = []
partes_panoid = []
inventario = []
esquema = None
for archivo in tqdm(archivos, desc="Leyendo países"):
    with pq.ParquetFile(archivo) as parquet:
        if not {"country_code", "panoid"}.issubset(parquet.schema_arrow.names):
            raise ValueError(f"Falta country_code o panoid en {archivo}")
        if esquema is None:
            esquema = parquet.schema_arrow
        elif not esquema.equals(parquet.schema_arrow, check_metadata=True):
            raise ValueError(f"El esquema difiere del primer archivo: {archivo}")
        metadatos = parquet.read(columns=["country_code", "panoid"])
        partes.append(metadatos.column("country_code").to_pandas())
        partes_panoid.append(metadatos.column("panoid").to_pandas())
        inventario.append({
            "archivo": str(archivo),
            "filas": parquet.metadata.num_rows,
            "bytes": archivo.stat().st_size,
            "mtime_ns": archivo.stat().st_mtime_ns,
        })

paises = pd.concat(partes, ignore_index=True).astype("string").str.strip().str.upper()
paises = paises.replace("", pd.NA)
panoids = pd.concat(partes_panoid, ignore_index=True).astype("string").str.strip().replace("", pd.NA)
del partes, partes_panoid
conteos = paises.value_counts().sort_index()
conteos.index.name = "country_code"
print(f"Filas totales: {len(paises):,}; sin país: {paises.isna().sum():,}")


def asignar_cupos(conteos, tamano, paises_admitidos, balanceado=False):
    """Cupos iguales (+/- 1) o proporcionales, sin reemplazo."""
    elegibles = conteos[conteos.index.isin(paises_admitidos) & (conteos > 0)].sort_index().astype("int64")
    disponible = int(elegibles.sum())
    if disponible == 0:
        raise ValueError("No hay filas locales de los paises admitidos por el CSV.")
    if tamano > disponible:
        raise ValueError(f"Se pidieron {tamano:,} filas pero solo hay {disponible:,} elegibles.")
    if balanceado:
        base, resto = divmod(tamano, len(elegibles))
        pueden_recibir_extra = elegibles.index[elegibles > base]
        if (elegibles < base).any() or len(pueden_recibir_extra) < resto:
            minimo_local = int(elegibles.min())
            maximo_balanceado = minimo_local * len(elegibles) + int((elegibles > minimo_local).sum())
            raise ValueError(
                f"No hay filas suficientes para balancear {tamano:,} muestras sin reemplazo. "
                f"El maximo balanceado con estos paises es {maximo_balanceado:,}. "
                "Reduci TAMANO_MUESTRA o usa BALANCEADO = False."
            )
        cupos = pd.Series(base, index=elegibles.index, dtype="int64")
        # Empates resueltos por codigo de pais para un resultado reproducible.
        cupos.loc[pueden_recibir_extra[:resto]] += 1
        return cupos[cupos > 0]
    numeradores = elegibles * tamano
    cupos = numeradores // disponible
    faltantes = tamano - int(cupos.sum())
    orden = (numeradores % disponible).sort_values(ascending=False, kind="stable")
    cupos.loc[orden.index[:faltantes]] += 1
    return cupos[cupos > 0]


cupos = asignar_cupos(conteos, TAMANO_MUESTRA, paises_admitidos, BALANCEADO)
modo_muestreo = "balanceado" if BALANCEADO else "proporcional"
print(f"Muestreo: {modo_muestreo}")
resumen = pd.concat([
    conteos_csv.rename("cantidad_csv"), conteos.rename("disponibles")
], axis=1).sort_index()
resumen.index.name = "country_code"
resumen["disponibles"] = resumen["disponibles"].fillna(0).astype("int64")
resumen["admitido_csv"] = resumen.index.isin(paises_admitidos)
resumen["seleccionadas"] = cupos.reindex(resumen.index, fill_value=0)
resumen["incluido"] = resumen["seleccionadas"] > 0
resumen["motivo"] = np.select(
    [resumen["cantidad_csv"].isna(), ~resumen["admitido_csv"],
     resumen["disponibles"] == 0, ~resumen["incluido"]],
    ["ausente del CSV", "cantidad CSV menor al umbral", "sin filas locales", "cupo cero"],
    default="incluido",
)
display(resumen)
print(f"Paises representados en la muestra: {len(cupos)}")
print(f"Filas de salida: {cupos.sum():,}")

Leyendo países: 100%|██████████| 393/393 [00:09<00:00, 39.91it/s]


Filas totales: 1,999,060; sin país: 7,932
Muestreo: balanceado


,cantidad_csv,disponibles,admitido_csv,seleccionadas,incluido,motivo
country_code,,,,,,
AD,24,24,False,0,False,cantidad CSV menor al umbral
AE,780,780,False,0,False,cantidad CSV menor al umbral
AL,624,624,False,0,False,cantidad CSV menor al umbral
AR,29668,29668,True,282,True,incluido
AT,9096,9096,True,282,True,incluido
...,...,...,...,...,...,...
US,476840,476840,True,281,True,incluido
UY,3000,3000,True,281,True,incluido
VI,44,44,False,0,False,cantidad CSV menor al umbral


Paises representados en la muestra: 71
Filas de salida: 20,000


In [21]:
rng = np.random.default_rng(SEMILLA)
grupos = paises.groupby(paises, sort=True).indices
indices = np.sort(np.concatenate([
    rng.choice(grupos[pais], size=int(cantidad), replace=False)
    for pais, cantidad in cupos.items()
])).astype(np.int64)
del grupos

conteos_seleccionados = paises.iloc[indices].value_counts().sort_index()
assert len(indices) == TAMANO_MUESTRA
assert len(np.unique(indices)) == TAMANO_MUESTRA
assert conteos_seleccionados.to_dict() == cupos.to_dict()

def separar_por_panorama(indices, paises, panoids, fraccion, semilla):
    """Separacion aleatoria por pais; cada panoid pertenece a una sola particion."""
    seleccion = pd.DataFrame({
        "indice": indices,
        "pais": paises.iloc[indices].to_numpy(),
        "panoid": panoids.iloc[indices].to_numpy(),
    })
    if seleccion["panoid"].isna().any():
        raise ValueError("Hay panoid faltantes en la muestra; no se puede garantizar una separacion por panorama.")
    if (seleccion.groupby("panoid")["pais"].nunique() > 1).any():
        raise ValueError("Hay panoramas asociados a varios paises en la muestra. Revisa la fuente.")
    rng_split = np.random.default_rng(semilla)
    validacion = []
    paises_sin_validacion = []
    for pais, filas in seleccion.groupby("pais", sort=True):
        tamanos = filas.groupby("panoid", sort=True).size()
        if len(tamanos) < 2:
            # Un unico panorama no puede estar en ambas particiones: queda en train.
            paises_sin_validacion.append(pais)
            continue
        orden = rng_split.permutation(len(tamanos))
        acumuladas = np.cumsum(tamanos.to_numpy()[orden])
        # Elegir el prefijo aleatorio mas cercano al objetivo, reservando un grupo para train.
        corte = int(np.argmin(np.abs(acumuladas[:-1] - len(filas) * fraccion))) + 1
        grupos_val = tamanos.index[orden[:corte]]
        validacion.extend(filas.loc[filas["panoid"].isin(grupos_val), "indice"].tolist())
    indices_val = np.sort(np.asarray(validacion, dtype=np.int64))
    indices_train = np.setdiff1d(indices, indices_val, assume_unique=True)
    if not len(indices_train) or not len(indices_val):
        raise ValueError("La muestra no permite train y validacion no vacios por panorama. Aumenta TAMANO_MUESTRA.")
    assert len(indices_train) + len(indices_val) == len(indices)
    assert not set(panoids.iloc[indices_train]) & set(panoids.iloc[indices_val])
    return indices_train, indices_val, paises_sin_validacion


indices_train, indices_val, paises_sin_validacion = separar_por_panorama(
    indices, paises, panoids, FRACCION_VALIDACION, SEMILLA
)
for nombre, seleccion in (("train", indices_train), ("validation", indices_val)):
    conteo = paises.iloc[seleccion].value_counts()
    resumen[nombre] = conteo.reindex(resumen.index, fill_value=0).astype("int64")
assert (resumen["train"] + resumen["validation"]).equals(resumen["seleccionadas"])
print(f"Train: {len(indices_train):,}; validacion: {len(indices_val):,} ({len(indices_val) / len(indices):.2%})")
if paises_sin_validacion:
    print(f"Paises con un unico panorama seleccionado, solo en train: {paises_sin_validacion}")
display(resumen)

porcentaje_val_nombre = format(FRACCION_VALIDACION * 100, ".12g")
nombre_muestra = (
    f"muestra_n-{TAMANO_MUESTRA}_umbral-csv-{MIN_FILAS_PAIS}"
    f"_{modo_muestreo}_paises-{len(cupos)}_seed-{SEMILLA}"
    f"_val-{porcentaje_val_nombre}pct_por-panoid"
)
ruta_salida = raiz_salida / nombre_muestra
if ruta_salida.exists():
    raise FileExistsError(f"La salida ya existe: {ruta_salida}. Cambia la configuracion o CARPETA_SALIDA.")
print(f"Salida: {ruta_salida}")

Train: 16,018; validacion: 3,982 (19.91%)


,cantidad_csv,disponibles,admitido_csv,seleccionadas,incluido,motivo,train,validation
country_code,,,,,,,,
AD,24,24,False,0,False,cantidad CSV menor al umbral,0,0
AE,780,780,False,0,False,cantidad CSV menor al umbral,0,0
AL,624,624,False,0,False,cantidad CSV menor al umbral,0,0
AR,29668,29668,True,282,True,incluido,226,56
AT,9096,9096,True,282,True,incluido,226,56
...,...,...,...,...,...,...,...,...
US,476840,476840,True,281,True,incluido,225,56
UY,3000,3000,True,281,True,incluido,225,56
VI,44,44,False,0,False,cantidad CSV menor al umbral,0,0


Salida: C:\Users\jurad\Documents\facultad\rrnn2\tp-final\geo-project\datasets\samples\muestra_n-20000_umbral-csv-1000_balanceado_paises-71_seed-42_val-20pct_por-panoid


## Guardar train y validacion

Se crean exactamente dos Parquet: `data/train.parquet` y `data/validation.parquet`, conservando todas las columnas y el esquema original. Las filas se escriben por lotes en orden de origen; la asignacion a las particiones es aleatoria y reproducible con `SEMILLA`. No se comparten filas ni panoramas entre train y validacion.

La carpeta incluye `resumen_paises.csv` con los conteos por particion, los indices globales de la muestra y de cada particion, y `config.json` con los porcentajes solicitado y real. El nombre de carpeta contiene el porcentaje solicitado. No se sobrescriben salidas anteriores. `COMPLETO` se crea solo al terminar; una ejecucion interrumpida deja una carpeta incompleta.

Para cargar ambos conjuntos: `load_dataset("parquet", data_files={"train": str(ruta_salida / "data/train.parquet"), "validation": str(ruta_salida / "data/validation.parquet")})`. Las imagenes embebidas se copian sin decodificarlas; las rutas a imagenes externas, si existen en la fuente, no se convierten en bytes.

In [22]:
def guardar_muestra(archivos, inventario, indices_train, indices_val, esquema, destino, tamano_lote):
    # Comprobar la fuente antes de crear una carpeta nueva.
    for archivo, info in zip(archivos, inventario, strict=True):
        estado = archivo.stat()
        if estado.st_size != info["bytes"] or estado.st_mtime_ns != info["mtime_ns"]:
            raise RuntimeError(f"La fuente cambio desde el conteo: {archivo}. Reejecuta desde el inicio.")
    destino.mkdir(parents=True, exist_ok=False)
    (destino / "data").mkdir()
    indices_totales = np.sort(np.concatenate([indices_train, indices_val]))
    escritores = {}
    guardadas = {"train": 0, "validation": 0}
    inicio_archivo = 0
    try:
        for nombre in guardadas:
            escritores[nombre] = pq.ParquetWriter(destino / "data" / f"{nombre}.parquet", esquema, compression="snappy")
        for archivo, info in tqdm(list(zip(archivos, inventario, strict=True)), desc="Guardando train y validacion"):
            fin_archivo = inicio_archivo + info["filas"]
            seleccion = indices_totales[np.searchsorted(indices_totales, inicio_archivo):np.searchsorted(indices_totales, fin_archivo)]
            if len(seleccion):
                with pq.ParquetFile(archivo) as parquet:
                    inicio_grupo = inicio_archivo
                    for grupo in range(parquet.num_row_groups):
                        fin_grupo = inicio_grupo + parquet.metadata.row_group(grupo).num_rows
                        if np.searchsorted(seleccion, inicio_grupo) < np.searchsorted(seleccion, fin_grupo):
                            inicio_lote = inicio_grupo
                            for lote in parquet.iter_batches(batch_size=tamano_lote, row_groups=[grupo]):
                                fin_lote = inicio_lote + lote.num_rows
                                globales = seleccion[np.searchsorted(seleccion, inicio_lote):np.searchsorted(seleccion, fin_lote)]
                                if len(globales):
                                    es_val = np.isin(globales, indices_val)
                                    for nombre, mascara in (("train", ~es_val), ("validation", es_val)):
                                        posiciones = globales[mascara] - inicio_lote
                                        if len(posiciones):
                                            tabla = pa.Table.from_batches([lote]).take(pa.array(posiciones))
                                            escritores[nombre].write_table(tabla)
                                            guardadas[nombre] += len(posiciones)
                                inicio_lote = fin_lote
                        inicio_grupo = fin_grupo
            inicio_archivo = fin_archivo
    finally:
        for escritor in escritores.values():
            escritor.close()
    esperado = {"train": len(indices_train), "validation": len(indices_val)}
    if guardadas != esperado:
        raise RuntimeError(f"Filas guardadas {guardadas}; se esperaban {esperado}.")
    return guardadas


guardadas = guardar_muestra(
    archivos, inventario, indices_train, indices_val, esquema, ruta_salida, TAMANO_LOTE
)
resumen.to_csv(ruta_salida / "resumen_paises.csv")
np.save(ruta_salida / "indices_seleccionados.npy", indices, allow_pickle=False)
np.save(ruta_salida / "indices_train.npy", indices_train, allow_pickle=False)
np.save(ruta_salida / "indices_validation.npy", indices_val, allow_pickle=False)
configuracion = {
    "tamano_muestra": TAMANO_MUESTRA,
    "fraccion_validacion_solicitada": FRACCION_VALIDACION,
    "fraccion_validacion_real": len(indices_val) / len(indices),
    "filas_train": len(indices_train),
    "filas_validation": len(indices_val),
    "separacion": "aleatoria por panoid dentro de cada pais, porcentaje aproximado",
    "paises_sin_validacion": paises_sin_validacion,
    "archivos_salida": {"train": "data/train.parquet", "validation": "data/validation.parquet"},
    "min_filas_pais_csv": MIN_FILAS_PAIS,
    "csv_conteos": str(ruta_csv),
    "conteos_csv": {pais: int(cantidad) for pais, cantidad in conteos_csv.items()},
    "paises_admitidos_csv": paises_admitidos.tolist(),
    "semilla": SEMILLA,
    "balanceado": BALANCEADO,
    "paises_incluidos": cupos.index.tolist(),
    "metodo": f"filtro por cantidad CSV + muestreo {modo_muestreo}",
    "unidad": "fila/imagen",
    "ruta_datos": str(ruta_datos),
    "patron_archivos": PATRON_ARCHIVOS,
    "archivos_fuente": inventario,
    "filas_fuente": len(paises),
    "filas_sin_pais": int(paises.isna().sum()),
}
(ruta_salida / "config.json").write_text(
    json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8"
)
(ruta_salida / "COMPLETO").write_text(f"{sum(guardadas.values())} filas guardadas: {guardadas}.\n", encoding="utf-8")
print(f"Muestra guardada: {sum(guardadas.values()):,} imágenes en {ruta_salida}")

Guardando train y validacion: 100%|██████████| 393/393 [11:33<00:00,  1.77s/it]


Muestra guardada: 20,000 imágenes en C:\Users\jurad\Documents\facultad\rrnn2\tp-final\geo-project\datasets\samples\muestra_n-20000_umbral-csv-1000_balanceado_paises-71_seed-42_val-20pct_por-panoid
